# Wall Assignment — pure-SAM method (stage 2 of 2)

**Method: SAM (no geometric prior).** Second stage of the **pure-SAM** branch: take the room
masks from `notebook_1_sam_auto_segmentation.ipynb` (`stage_sam_auto`) and assign each room
its wall geometry, then back-project to 3-D. This is the **identical boundary-ring wall
assignment** used by `methods/geometric/notebook_2_wall_assignment.ipynb` — only the *mask
source* differs (SAM auto-segmentation masks instead of watershed masks), so the three
methods are compared end-to-end (segmentation **and** resulting walls).

## Inputs
- `wallness.npy` + `transform.json` from `stage1_occupancy.zip` (the wall source).
- `room_labels.npy` from `stage_sam_auto.zip` (the room masks `M_i`).
- The **point cloud** from `CFG.file_path` (reloaded with the original loader; deterministic).

## Outputs  (`{out_root}/stage_sam_walls/`, zipped to `stage_sam_walls.zip`)
- `room_XX_walls.ply` — per-room 3-D wall points.
- `room_wall_masks.npz` — per-room wall-pixel masks (keyed `room_<id>`).
- `room_labels.npy` (the masks used), `transform.json`, `config.json`.

### Setup
**Run-All ready.** Edit **`params.yaml`** (the only config surface), then run every cell top
to bottom. This is a **CPU** stage; run it locally after downloading `stage_sam_auto.zip`
from Colab into your `scan2bim_out/`.

In [ ]:
# ============================== scan2bim setup (local) ==============================
# Thin driver: all logic lives in the `scan2bim/` package. With `pip install -e .`,
# `import scan2bim` works from anywhere; `load_config()` reads params.yaml (the ONLY file
# you edit) over the Config defaults and resolves paths.
import os
import numpy as np
import scan2bim
from scan2bim import artifacts as A, viz

CFG = scan2bim.load_config()        # params.yaml over Config defaults; file_path/out_root -> abs
SHOW_DEBUG = True                   # set False to skip the QA plots
print('scan2bim', scan2bim.__version__, 'loaded from', os.path.dirname(scan2bim.__file__))
print('input cloud :', CFG.file_path, '| exists:', os.path.isfile(CFG.file_path))
print('output root :', CFG.out_root)

### Step 1 — Load inputs (SAM auto-segmentation masks)
Room masks come from the **pure-SAM** stage 1 (`stage_sam_auto`). The guards below fail loudly
if an upstream stage was produced from a different cloud / grid.

In [ ]:
s1 = A.load_stage_dir(CFG.out_root, A.STAGE1)
wallness = A.load_npy(os.path.join(s1, A.WALLNESS_NPY)).astype(bool)
tf = A.load_transform(os.path.join(s1, A.TRANSFORM_JSON))

# Room masks come from the pure-SAM method's stage-1 output (stage_sam_auto).
s_sam = A.load_stage_dir(CFG.out_root, A.STAGE_SAM_AUTO)
scan2bim.assert_upstream_config(CFG, A.load_stage_config(s1))      # same cloud + grid as stage 1
scan2bim.assert_upstream_config(CFG, A.load_stage_config(s_sam))   # ...and the SAM stage
labels = A.load_npy(os.path.join(s_sam, A.ROOM_LABELS_NPY)).astype('int32')
print('room masks from: SAM auto (stage_sam_auto) | rooms:',
      len([r for r in np.unique(labels) if r >= 1]))

# Reload the cloud with the SAME loader N1 used (deterministic -> aligns to tf), then assert
# it actually lands inside the stage-1 grid.
pcd, pts = scan2bim.load_point_cloud(CFG)
scan2bim.assert_points_in_grid(pts, tf)

### Step 2 — Boundary-ring wall masks (identical to the geometric method)

In [ ]:
wall_masks, dbg = scan2bim.room_wall_masks_boundary_ring(labels, wallness, CFG, return_debug=True)
print('rooms:', len(wall_masks), '| erode_px=%d  r_w(dilate_px)=%d' % (dbg['erode_px'], dbg['dilate_px']))

### Step 3 — Back-project the assigned wall pixels into 3-D

In [ ]:
band, floor_z, ceil_z = scan2bim.height_band_mask(pts, CFG, tf)
rooms3d = scan2bim.backproject_room_masks(pts, wall_masks, tf, keep_mask=band)
for e in rooms3d:
    print('room %02d: %7d wall points' % (e['room_id'], len(e['points'])))

### Optional — QA plot of the boundary-ring assignment

In [ ]:
if SHOW_DEBUG:
    viz.show_wall_assignment(labels, wall_masks, debug=dbg)

### Step 4 — Save per-room wall clouds + masks and package the ZIP

In [ ]:
import open3d as o3d
out_dir = A.ensure_dir(A.stage_dir(CFG.out_root, A.STAGE_SAM_WALLS))
A.save_room_wall_masks(os.path.join(out_dir, A.ROOM_WALL_MASKS_NPZ), wall_masks)

n_written = 0
for e in rooms3d:
    if len(e['points']) == 0:
        continue
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(e['points'])
    o3d.io.write_point_cloud(os.path.join(out_dir, 'room_%02d_walls.ply' % e['room_id']), pc)
    n_written += 1
print('wrote', n_written, 'room wall clouds')

A.save_npy(os.path.join(out_dir, A.ROOM_LABELS_NPY), labels)
A.save_transform(os.path.join(out_dir, A.TRANSFORM_JSON), tf,
                 extra=dict(floor_z=float(floor_z), ceil_z=float(ceil_z)))
A.save_config(os.path.join(out_dir, A.CONFIG_JSON), CFG)
zip_path = A.package_stage(CFG.out_root, A.STAGE_SAM_WALLS)
print('packaged ->', zip_path)

### Optional — download the wall-cloud ZIP (Colab)

In [ ]:
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not in Colab - find the ZIP at:', zip_path)